# ETAPA: Auditoria GATE 0 (Read-Only) — v2

- Regra: leitura apenas; permitido iniciar Docker/MinIO **somente para consulta/listagem** (sem escrita em dados).
- Notebook para auditoria de infra/buckets/transferências conforme brief v2.
- Timestamp de início será registrado na próxima célula.



In [1]:
from datetime import datetime, timezone
print("timestamp_utc", datetime.now(timezone.utc).isoformat())



timestamp_utc 2025-12-31T14:11:59.223753+00:00


## Seção A — Auditoria do repo (Git + estrutura)
- Confirma git/branch/staged.
- Verifica caminhos reais de infra e artefatos locais.
- Localiza TESTE1.ipynb citado.



In [2]:
import subprocess, os
from pathlib import Path


def run(cmd, cwd=None):
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    return {
        "cmd": cmd,
        "rc": res.returncode,
        "out": res.stdout.strip(),
        "err": res.stderr.strip(),
    }

repo_root_path = Path(run("git rev-parse --show-toplevel")["out"] or ".").resolve()
status = run("git status -sb", cwd=repo_root_path)
staged_names = run("git diff --name-only --staged", cwd=repo_root_path)
staged_stat = run("git diff --stat --staged", cwd=repo_root_path)

paths_to_check = {
    "infra/minio_operacao.md": repo_root_path / "fase_1_diagnostico/infra/minio_operacao.md",
    "infra/ambiente_ds-base.md": repo_root_path / "fase_1_diagnostico/infra/ambiente_ds-base.md",
    "compose_minio_dir": repo_root_path / "fase_1_diagnostico/infra/minio",
    "compose_minio_file": repo_root_path / "fase_1_diagnostico/infra/minio/docker-compose.yml",
    "minio_credentials_env": repo_root_path / "fase_1_diagnostico/infra/env/minio_credentials.env",
    "dados_iniciais": repo_root_path / "fase_1_diagnostico/dados/dados_iniciais",
    "manifestos_inicial": repo_root_path / "fase_1_diagnostico/dados/manifestos_inicial",
    "manifestos_final": repo_root_path / "fase_1_diagnostico/dados/manifestos_final",
}

csvs = {
    "inventario_dados_iniciais.csv": repo_root_path / "fase_1_diagnostico/dados/manifestos_inicial/inventario_dados_iniciais.csv",
    "manifest_intermediario.csv": repo_root_path / "fase_1_diagnostico/dados/manifestos_inicial/manifest_intermediario.csv",
    "manifest_arquivos_classificados.csv": repo_root_path / "fase_1_diagnostico/dados/manifestos_final/manifest_arquivos_classificados.csv",
    "manifest_unificado.csv": repo_root_path / "fase_1_diagnostico/dados/manifestos_final/manifest_unificado.csv",
}

# localizar TESTE1.ipynb
teste1_matches = list(repo_root_path.glob("**/TESTE1.ipynb"))

print(f"repo_root: {repo_root_path}")
print("git status -sb:\n", status["out"])
print("staged names:\n", staged_names["out"] or "(nenhum)")
print("staged stat:\n", staged_stat["out"] or "(nenhum)")

print("\nExistência de paths principais:")
for label, p in paths_to_check.items():
    print(f"- {label}: {'ENCONTRADO' if p.exists() else 'NAO_ENCONTRADO'} -> {p}")

print("\nCSVs alvo:")
for name, path in csvs.items():
    print(f"- {name}: {'ENCONTRADO' if path.exists() else 'NAO_ENCONTRADO'} -> {path}")

print("\nNotebook TESTE1.ipynb (busca recursiva):")
if teste1_matches:
    for m in teste1_matches:
        print(f"- ENCONTRADO: {m}")
else:
    print("- NAO_ENCONTRADO")



repo_root: /home/wilson/Maringa
git status -sb:
 ## main...origin/main
?? fase_1_diagnostico/analise/RETOMADA_GATE0_AUDITORIA_MINIO.ipynb
staged names:
 (nenhum)
staged stat:
 (nenhum)

Existência de paths principais:
- infra/minio_operacao.md: ENCONTRADO -> /home/wilson/Maringa/fase_1_diagnostico/infra/minio_operacao.md
- infra/ambiente_ds-base.md: ENCONTRADO -> /home/wilson/Maringa/fase_1_diagnostico/infra/ambiente_ds-base.md
- compose_minio_dir: ENCONTRADO -> /home/wilson/Maringa/fase_1_diagnostico/infra/minio
- compose_minio_file: ENCONTRADO -> /home/wilson/Maringa/fase_1_diagnostico/infra/minio/docker-compose.yml
- minio_credentials_env: ENCONTRADO -> /home/wilson/Maringa/fase_1_diagnostico/infra/env/minio_credentials.env
- dados_iniciais: ENCONTRADO -> /home/wilson/Maringa/fase_1_diagnostico/dados/dados_iniciais
- manifestos_inicial: ENCONTRADO -> /home/wilson/Maringa/fase_1_diagnostico/dados/manifestos_inicial
- manifestos_final: ENCONTRADO -> /home/wilson/Maringa/fase_1_diagnos

## Seção B — Docker (detectar e iniciar se necessário)
- Detecta CLI/daemon.
- Tenta iniciar se não responder; se exigir intervenção, marca INCONCLUSIVO.



In [3]:
docker_status = {
    "cli_present": False,
    "daemon_ok": False,
    "message": "",
    "ps_output": None,
}


def safe_run(cmd, cwd=None):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)

# Verifica CLI
res_cli = safe_run("docker --version")
if res_cli.returncode == 0:
    docker_status["cli_present"] = True
else:
    docker_status["message"] = f"docker CLI indisponivel: rc={res_cli.returncode}, err={res_cli.stderr.strip()}"

# Verifica daemon
if docker_status["cli_present"]:
    res_ps = safe_run("docker ps --format 'table {{.Names}}\t{{.Image}}\t{{.Status}}\t{{.Ports}}'")
    if res_ps.returncode == 0:
        docker_status["daemon_ok"] = True
        docker_status["ps_output"] = res_ps.stdout
    else:
        # tenta iniciar serviço (pode falhar se exigir sudo)
        try_start = safe_run("sudo systemctl start docker")
        if try_start.returncode == 0:
            res_ps = safe_run("docker ps --format 'table {{.Names}}\t{{.Image}}\t{{.Status}}\t{{.Ports}}'")
            if res_ps.returncode == 0:
                docker_status["daemon_ok"] = True
                docker_status["ps_output"] = res_ps.stdout
            else:
                docker_status["message"] = f"docker daemon inacessivel apos start: rc={res_ps.returncode}, err={res_ps.stderr.strip()}"
        else:
            docker_status["message"] = f"docker daemon inacessivel e start falhou (sudo systemctl start docker): rc={try_start.returncode}, err={try_start.stderr.strip()}"

print("docker_status:", docker_status)
if docker_status.get("ps_output"):
    print("\ndocker ps:\n" + docker_status["ps_output"])



docker_status: {'cli_present': True, 'daemon_ok': True, 'message': '', 'ps_output': 'NAMES           IMAGE                                      STATUS       PORTS\nmaringa-minio   minio/minio:RELEASE.2025-09-07T16-13-09Z   Up 3 hours   0.0.0.0:9000-9001->9000-9001/tcp, [::]:9000-9001->9000-9001/tcp\n'}

docker ps:
NAMES           IMAGE                                      STATUS       PORTS
maringa-minio   minio/minio:RELEASE.2025-09-07T16-13-09Z   Up 3 hours   0.0.0.0:9000-9001->9000-9001/tcp, [::]:9000-9001->9000-9001/tcp



## Seção C — MinIO (detectar e iniciar via compose)
- Usa compose em `fase_1_diagnostico/infra/minio` se existir.
- Pode subir `docker compose up -d` para permitir listagem.



In [4]:
minio_info = {
    "status": "INCONCLUSIVO",
    "containers": [],
    "compose_present": (repo_root_path / "fase_1_diagnostico/infra/minio/docker-compose.yml").exists(),
    "logs": None,
    "message": "",
}

compose_dir = repo_root_path / "fase_1_diagnostico/infra/minio"
compose_file = compose_dir / "docker-compose.yml"

if docker_status.get("daemon_ok"):
    # Verifica containers já rodando
    ps_lines = docker_status.get("ps_output", "").strip().splitlines()
    for line in ps_lines[1:]:  # skip header
        parts = line.split()
        if not parts:
            continue
        name = parts[0]
        image = parts[1] if len(parts) > 1 else ""
        status_c = parts[2] if len(parts) > 2 else ""
        ports = " ".join(parts[3:]) if len(parts) > 3 else ""
        if "minio" in name.lower() or "minio" in image.lower():
            minio_info["containers"].append({"name": name, "image": image, "status": status_c, "ports": ports})

    # Se não há container e existe compose, tentar subir
    if not minio_info["containers"] and compose_file.exists():
        up = safe_run("docker compose up -d", cwd=compose_dir)
        if up.returncode == 0:
            res_ps = safe_run("docker ps --format 'table {{.Names}}\t{{.Image}}\t{{.Status}}\t{{.Ports}}'")
            if res_ps.returncode == 0:
                minio_info["ps_output_after_up"] = res_ps.stdout
                ps_lines = res_ps.stdout.strip().splitlines()
                for line in ps_lines[1:]:
                    parts = line.split()
                    if not parts:
                        continue
                    name = parts[0]
                    image = parts[1] if len(parts) > 1 else ""
                    status_c = parts[2] if len(parts) > 2 else ""
                    ports = " ".join(parts[3:]) if len(parts) > 3 else ""
                    if "minio" in name.lower() or "minio" in image.lower():
                        minio_info["containers"].append({"name": name, "image": image, "status": status_c, "ports": ports})
            else:
                minio_info["message"] = f"compose up ok, mas docker ps falhou: rc={res_ps.returncode}"
        else:
            minio_info["message"] = f"falha ao subir compose: rc={up.returncode}, err={up.stderr.strip()}"

    if minio_info["containers"]:
        minio_info["status"] = "UP"
        cname = minio_info["containers"][0]["name"]
        logs = safe_run(f"docker logs --tail 40 {cname}")
        if logs.returncode == 0:
            minio_info["logs"] = "\n".join(logs.stdout.strip().splitlines()[:40])
        else:
            minio_info["message"] = f"falha ao ler logs: rc={logs.returncode}"
    else:
        minio_info["status"] = "DOWN"
        if minio_info["compose_present"]:
            minio_info["message"] = "compose presente mas container nao subiu"
        else:
            minio_info["message"] = "compose nao encontrado"

print("minio_info:", {k: v for k, v in minio_info.items() if k != "logs"})
if minio_info.get("logs"):
    print("\nlogs (tail):\n" + minio_info["logs"])



minio_info: {'status': 'UP', 'containers': [{'name': 'maringa-minio', 'image': 'minio/minio:RELEASE.2025-09-07T16-13-09Z', 'status': 'Up', 'ports': '3 hours 0.0.0.0:9000-9001->9000-9001/tcp, [::]:9000-9001->9000-9001/tcp'}], 'compose_present': True, 'message': ''}


## Seção D — Endpoint e credenciais (sem vazar valores)
- Lê `infra/env/minio_credentials.env` se existir (sem imprimir valores).
- Caso contrário, tenta variáveis de ambiente.



In [5]:
from typing import Optional

cred_info = {
    "endpoint": None,
    "access_key_present": False,
    "secret_key_present": False,
    "source": "",
    "note": "",
}

env_file = repo_root_path / "fase_1_diagnostico/infra/env/minio_credentials.env"
endpoint = None
access_key = None
secret_key = None

# 1) tentar ler env file (sem imprimir valores)
if env_file.exists():
    lines = env_file.read_text().splitlines()
    for line in lines:
        if not line or line.strip().startswith("#"):
            continue
        if "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip()
        if k == "MINIO_ROOT_USER" or k == "MINIO_ACCESS_KEY":
            access_key = v
        elif k == "MINIO_ROOT_PASSWORD" or k == "MINIO_SECRET_KEY":
            secret_key = v
        elif k == "MINIO_SERVER_PORT":
            endpoint = f"http://localhost:{v}"
    cred_info["source"] = ".env file"

# 2) fallback para variáveis de ambiente
if not access_key:
    access_key = os.getenv("MINIO_ACCESS_KEY") or os.getenv("MINIO_ROOT_USER")
if not secret_key:
    secret_key = os.getenv("MINIO_SECRET_KEY") or os.getenv("MINIO_ROOT_PASSWORD")
if not endpoint:
    port_env = os.getenv("MINIO_SERVER_PORT")
    endpoint = f"http://localhost:{port_env}" if port_env else "http://localhost:9000"

cred_info["endpoint"] = endpoint
cred_info["access_key_present"] = bool(access_key)
cred_info["secret_key_present"] = bool(secret_key)

print("fonte creds:", cred_info["source"] or "variaveis de ambiente/default")
print("endpoint definido (mascarado):", "ENCONTRADO" if endpoint else "NAO" )
print("access_key presente:", cred_info["access_key_present"])
print("secret_key presente:", cred_info["secret_key_present"])
if not (cred_info["access_key_present"] and cred_info["secret_key_present"]):
    cred_info["note"] = "Faltam chaves; auditoria S3 pode ficar inconclusiva."
print("nota:", cred_info["note"])



fonte creds: .env file
endpoint definido (mascarado): ENCONTRADO
access_key presente: True
secret_key presente: True
nota: 


## Seção E — Auditoria S3/MinIO (somente leitura, cliente `minio`)
- Lista buckets e gera stats (contagem, tamanho, amostra até 20).



In [6]:
import math
try:
    from minio import Minio
except Exception as e:
    Minio = None

s3_audit = {
    "status": "INCONCLUSIVO",
    "reason": "",
    "buckets": [],
    "bucket_stats": {},
    "local_dados_iniciais_count": None,
}

# contar arquivos locais em dados_iniciais
dados_iniciais_path = repo_root_path / "fase_1_diagnostico/dados/dados_iniciais"
local_count = sum(len(files) for _, _, files in os.walk(dados_iniciais_path)) if dados_iniciais_path.exists() else 0
s3_audit["local_dados_iniciais_count"] = local_count

endpoint = cred_info.get("endpoint")
access_key = access_key if 'access_key' in locals() else None
secret_key = secret_key if 'secret_key' in locals() else None

if not (access_key and secret_key):
    s3_audit["status"] = "INCONCLUSIVO"
    s3_audit["reason"] = "Credenciais ausentes"
    print(s3_audit)
elif Minio is None:
    s3_audit["status"] = "INCONCLUSIVO"
    s3_audit["reason"] = "pacote minio nao disponivel"
    print(s3_audit)
else:
    try:
        client = Minio(endpoint.replace("http://", ""), access_key=access_key, secret_key=secret_key, secure=False)
        buckets = client.list_buckets()
        s3_audit["buckets"] = [b.name for b in buckets]
        for b in s3_audit["buckets"]:
            total = 0
            size = 0
            sample = []
            for obj in client.list_objects(b, recursive=True):
                total += 1
                size += obj.size or 0
                if len(sample) < 20:
                    sample.append(obj.object_name)
            s3_audit["bucket_stats"][b] = {
                "n_objetos": total,
                "tamanho_bytes": size,
                "amostra": sample,
            }
        s3_audit["status"] = "OK"
    except Exception as e:
        s3_audit["status"] = "INCONCLUSIVO"
        s3_audit["reason"] = f"falha ao listar buckets: {e}"
    print(s3_audit)



{'status': 'OK', 'reason': '', 'buckets': ['maringa-derived', 'maringa-manifests', 'maringa-raw', 'maringa-refined'], 'bucket_stats': {'maringa-derived': {'n_objetos': 0, 'tamanho_bytes': 0, 'amostra': []}, 'maringa-manifests': {'n_objetos': 0, 'tamanho_bytes': 0, 'amostra': []}, 'maringa-raw': {'n_objetos': 0, 'tamanho_bytes': 0, 'amostra': []}, 'maringa-refined': {'n_objetos': 0, 'tamanho_bytes': 0, 'amostra': []}}, 'local_dados_iniciais_count': 145}


## Seção F — Evidência de transferências reais
- Usa stats dos buckets: maringa-raw / maringa-manifests.



In [7]:
transferencia = {
    "status_raw": "INCONCLUSIVO",
    "status_manifests": "INCONCLUSIVO",
    "infra_status": s3_audit.get("reason", ""),
    "notes": [],
}

if s3_audit.get("status") == "OK":
    stats = s3_audit.get("bucket_stats", {})
    if "maringa-raw" in stats:
        n = stats["maringa-raw"].get("n_objetos", 0)
        transferencia["status_raw"] = "TRANSFERENCIA_CONFIRMADA" if n > 0 else "INFRA_OK_SEM_MIGRACAO"
        transferencia["notes"].append(f"maringa-raw objetos={n}")
    if "maringa-manifests" in stats:
        n = stats["maringa-manifests"].get("n_objetos", 0)
        transferencia["status_manifests"] = "PUBLICACAO_CONFIRMADA" if n > 0 else "INFRA_OK_SEM_PUBLICACAO"
        transferencia["notes"].append(f"maringa-manifests objetos={n}")
else:
    transferencia["status_raw"] = "INCONCLUSIVO"
    transferencia["status_manifests"] = "INCONCLUSIVO"

print(transferencia)



{'status_raw': 'INFRA_OK_SEM_MIGRACAO', 'status_manifests': 'INFRA_OK_SEM_PUBLICACAO', 'infra_status': '', 'notes': ['maringa-raw objetos=0', 'maringa-manifests objetos=0']}


## Seção G — Matriz Documento vs Realidade
- Consolida hipóteses vs achados.



In [8]:
from pprint import pprint

def status_bool(ok):
    return "CONFIRMADO" if ok else "NAO_ENCONTRADO"

def status_inconclusive(msg="INCONCLUSIVO"):
    return "INCONCLUSIVO" if not msg else f"INCONCLUSIVO ({msg})"

matrix = []

matrix.append({
    "hipotese": "Docker daemon responde",
    "encontrado": docker_status.get("daemon_ok"),
    "status": status_bool(docker_status.get("daemon_ok")),
    "evidencia": "docker ps",
})

matrix.append({
    "hipotese": "MinIO rodando",
    "encontrado": minio_info.get("status") == "UP",
    "status": status_bool(minio_info.get("status") == "UP"),
    "evidencia": "docker ps / logs tail",
})

matrix.append({
    "hipotese": "Existe operacao MinIO documentada",
    "encontrado": paths_to_check["infra/minio_operacao.md"].exists(),
    "status": status_bool(paths_to_check["infra/minio_operacao.md"].exists()),
    "evidencia": str(paths_to_check["infra/minio_operacao.md"]),
})

matrix.append({
    "hipotese": "Compose MinIO presente no repo",
    "encontrado": paths_to_check["compose_minio_file"].exists(),
    "status": status_bool(paths_to_check["compose_minio_file"].exists()),
    "evidencia": str(paths_to_check["compose_minio_file"]),
})

matrix.append({
    "hipotese": "Credenciais presentes no filesystem",
    "encontrado": paths_to_check["minio_credentials_env"].exists(),
    "status": status_bool(paths_to_check["minio_credentials_env"].exists()),
    "evidencia": str(paths_to_check["minio_credentials_env"]),
})

# Buckets esperados
for bname in ["maringa-raw", "maringa-refined", "maringa-derived", "maringa-manifests"]:
    if s3_audit.get("status") == "OK":
        exists = bname in s3_audit.get("buckets", [])
        st = status_bool(exists)
    else:
        exists = None
        st = status_inconclusive(s3_audit.get("reason"))
    matrix.append({
        "hipotese": f"Bucket {bname} existe",
        "encontrado": exists,
        "status": st,
        "evidencia": "list_buckets" if s3_audit.get("status") == "OK" else s3_audit.get("reason"),
    })

# Transferencias
if s3_audit.get("status") == "OK":
    tr_status = transferencia.get("status_raw")
    matrix.append({
        "hipotese": "Transferencia real em maringa-raw",
        "encontrado": tr_status == "TRANSFERENCIA_CONFIRMADA",
        "status": tr_status,
        "evidencia": transferencia.get("notes"),
    })
    man_status = transferencia.get("status_manifests")
    matrix.append({
        "hipotese": "Manifesto publicado em maringa-manifests",
        "encontrado": man_status == "PUBLICACAO_CONFIRMADA",
        "status": man_status,
        "evidencia": transferencia.get("notes"),
    })
else:
    matrix.append({
        "hipotese": "Transferencia real em maringa-raw",
        "encontrado": None,
        "status": status_inconclusive(s3_audit.get("reason")),
        "evidencia": "sem listagem",
    })
    matrix.append({
        "hipotese": "Manifesto publicado em maringa-manifests",
        "encontrado": None,
        "status": status_inconclusive(s3_audit.get("reason")),
        "evidencia": "sem listagem",
    })

# Artefatos locais
matrix.append({
    "hipotese": "inventario_dados_iniciais.csv presente",
    "encontrado": csvs["inventario_dados_iniciais.csv"].exists(),
    "status": status_bool(csvs["inventario_dados_iniciais.csv"].exists()),
    "evidencia": str(csvs["inventario_dados_iniciais.csv"]),
})
matrix.append({
    "hipotese": "manifest_intermediario.csv presente",
    "encontrado": csvs["manifest_intermediario.csv"].exists(),
    "status": status_bool(csvs["manifest_intermediario.csv"].exists()),
    "evidencia": str(csvs["manifest_intermediario.csv"]),
})
matrix.append({
    "hipotese": "manifest_arquivos_classificados.csv presente",
    "encontrado": csvs["manifest_arquivos_classificados.csv"].exists(),
    "status": status_bool(csvs["manifest_arquivos_classificados.csv"].exists()),
    "evidencia": str(csvs["manifest_arquivos_classificados.csv"]),
})
matrix.append({
    "hipotese": "manifest_unificado.csv presente",
    "encontrado": csvs["manifest_unificado.csv"].exists(),
    "status": status_bool(csvs["manifest_unificado.csv"].exists()),
    "evidencia": str(csvs["manifest_unificado.csv"]),
})

matrix.append({
    "hipotese": "Notebook TESTE1.ipynb existe",
    "encontrado": len(teste1_matches) > 0,
    "status": status_bool(len(teste1_matches) > 0),
    "evidencia": ", ".join(str(p) for p in teste1_matches) or "NAO_ENCONTRADO",
})

pprint(matrix)



[{'encontrado': True,
  'evidencia': 'docker ps',
  'hipotese': 'Docker daemon responde',
  'status': 'CONFIRMADO'},
 {'encontrado': True,
  'evidencia': 'docker ps / logs tail',
  'hipotese': 'MinIO rodando',
  'status': 'CONFIRMADO'},
 {'encontrado': True,
  'evidencia': '/home/wilson/Maringa/fase_1_diagnostico/infra/minio_operacao.md',
  'hipotese': 'Existe operacao MinIO documentada',
  'status': 'CONFIRMADO'},
 {'encontrado': True,
  'evidencia': '/home/wilson/Maringa/fase_1_diagnostico/infra/minio/docker-compose.yml',
  'hipotese': 'Compose MinIO presente no repo',
  'status': 'CONFIRMADO'},
 {'encontrado': True,
  'evidencia': '/home/wilson/Maringa/fase_1_diagnostico/infra/env/minio_credentials.env',
  'hipotese': 'Credenciais presentes no filesystem',
  'status': 'CONFIRMADO'},
 {'encontrado': True,
  'evidencia': 'list_buckets',
  'hipotese': 'Bucket maringa-raw existe',
  'status': 'CONFIRMADO'},
 {'encontrado': True,
  'evidencia': 'list_buckets',
  'hipotese': 'Bucket marin

## Seção H — Relatório final (Sumário, Evidências essenciais, Matriz, Próximo passo)



In [9]:
def summarize():
    def get_out(obj):
        if isinstance(obj, dict):
            return obj.get("out") or obj.get("stdout") or str(obj)
        return str(obj)

    lines = []
    lines.append("### Sumário Executivo")
    lines.append(f"- Git: {get_out(status)}")
    staged_val = staged_names.get("out") if isinstance(staged_names, dict) else staged_names
    lines.append(f"- Staged: {staged_val or 'nenhum'}")
    docker_line = "UP" if docker_status.get("daemon_ok") else "DOWN/INCONCLUSIVO"
    lines.append(f"- Docker: {docker_line}")
    lines.append(f"- MinIO: {minio_info.get('status')} ({minio_info.get('message') or 'ok'})")
    if s3_audit.get("status") == "OK":
        lines.append(f"- Buckets: {s3_audit.get('buckets')}")
    else:
        lines.append(f"- Buckets: INCONCLUSIVO ({s3_audit.get('reason')})")
    lines.append(f"- Transferencias raw: {transferencia.get('status_raw')}")
    lines.append(f"- Publicacao manifests: {transferencia.get('status_manifests')}")

    lines.append("\n### Evidencias essenciais")
    lines.append("- docker ps (Seção B)")
    lines.append("- compose/log MinIO (Seção C)")
    lines.append("- caminhos/artefatos (Seção A)")
    lines.append("- auditoria S3 (Seção E)")

    lines.append("\n### Matriz Documento vs Realidade")
    for item in matrix:
        lines.append(f"- {item['hipotese']}: {item['status']} | evid: {item['evidencia']}")

    lines.append("\n### Proximo passo imediato")
    lines.append("- Se credenciais faltarem, disponibilizar MINIO_ROOT_USER / MINIO_ROOT_PASSWORD ou ACCESS/SECRET para listar buckets.")
    lines.append("- Reexecutar auditoria S3 (Seções E/F) após credenciais.")

    print("\n".join(lines))

summarize()



### Sumário Executivo
- Git: ## main...origin/main
?? fase_1_diagnostico/analise/RETOMADA_GATE0_AUDITORIA_MINIO.ipynb
- Staged: nenhum
- Docker: UP
- MinIO: UP (ok)
- Buckets: ['maringa-derived', 'maringa-manifests', 'maringa-raw', 'maringa-refined']
- Transferencias raw: INFRA_OK_SEM_MIGRACAO
- Publicacao manifests: INFRA_OK_SEM_PUBLICACAO

### Evidencias essenciais
- docker ps (Seção B)
- compose/log MinIO (Seção C)
- caminhos/artefatos (Seção A)
- auditoria S3 (Seção E)

### Matriz Documento vs Realidade
- Docker daemon responde: CONFIRMADO | evid: docker ps
- MinIO rodando: CONFIRMADO | evid: docker ps / logs tail
- Existe operacao MinIO documentada: CONFIRMADO | evid: /home/wilson/Maringa/fase_1_diagnostico/infra/minio_operacao.md
- Compose MinIO presente no repo: CONFIRMADO | evid: /home/wilson/Maringa/fase_1_diagnostico/infra/minio/docker-compose.yml
- Credenciais presentes no filesystem: CONFIRMADO | evid: /home/wilson/Maringa/fase_1_diagnostico/infra/env/minio_credentials.env


## Encerramento — Confirmação Git



In [10]:
print(run("git status -sb") )



{'cmd': 'git status -sb', 'rc': 0, 'out': '## main...origin/main\n?? RETOMADA_GATE0_AUDITORIA_MINIO.ipynb', 'err': ''}
